# Training Signal Analysis

Analyze loss landscape and training signal distribution: per-token loss,
component attribution, entropy profiles, concentration, and summary.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.training_signal_analysis import (
    per_token_loss_analysis, loss_component_attribution,
    prediction_entropy_profile, loss_concentration_analysis,
    training_signal_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Per-Token Loss

Which positions does the model find hardest to predict?

In [ ]:
result = per_token_loss_analysis(model, tokens)
print(f"Mean loss: {result['mean_loss']:.4f}")
print(f"Correct: {result['n_correct']}/{len(result['per_position'])}\n")
for p in result['per_position']:
    bar = '█' * int(min(p['loss'], 10))
    print(f"  Pos {p['position']}: loss={p['loss']:.3f}, prob={p['probability']:.4f}, "
          f"rank={p['rank']} {bar}")

## Loss Component Attribution

Which components help or hurt the prediction?

In [ ]:
result = loss_component_attribution(model, tokens)
print(f"Target token: {result['target_token']}")
print(f"Total attn: {result['total_attn']:+.4f}, Total mlp: {result['total_mlp']:+.4f}\n")
for p in result['per_layer']:
    flag = ' [HELPS]' if p['helps_prediction'] else ' [HURTS]'
    print(f"  Layer {p['layer']}: attn={p['attn_target_logit']:+.4f}, "
          f"mlp={p['mlp_target_logit']:+.4f}{flag}")

## Prediction Entropy Profile

How confident is the model at each position?

In [ ]:
result = prediction_entropy_profile(model, tokens)
print(f"Mean entropy: {result['mean_entropy']:.4f}")
print(f"Confident positions: {result['n_confident']}\n")
for p in result['per_position']:
    bar = '█' * int(p['max_prob'] * 30)
    conf = ' [CONF]' if p['is_confident'] else ''
    print(f"  Pos {p['position']}: H={p['entropy']:.3f}, "
          f"max_p={p['max_prob']:.4f}, top={p['top_prediction']}{conf} {bar}")

## Loss Concentration

Is the training signal concentrated on a few positions?

In [ ]:
result = loss_concentration_analysis(model, tokens)
print(f"Gini coefficient: {result['gini']:.4f}")
print(f"Top 20% loss fraction: {result['top_20pct_loss_fraction']:.2%}")
print(f"Concentrated: {result['is_concentrated']}")

## Training Signal Summary

Combined overview of loss, entropy, and prediction quality.

In [ ]:
result = training_signal_summary(model, tokens)
print(f"Mean loss: {result['mean_loss']:.4f}")
print(f"Mean entropy: {result['mean_entropy']:.4f}")
print(f"Correct: {result['n_correct']}, Confident: {result['n_confident']}\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: loss={p['loss']:.3f}, rank={p['rank']}, "
          f"H={p['entropy']:.3f}, max_p={p['max_prob']:.4f}")